# Modelagem e Comparação de Algoritmos - Wine Quality Classification

Este notebook faz parte do **Tech Challenge da Pós-Tech em Data Analytics (Fase 2)**. 
Neste momento, focamos na **Fase 3: Pré-processamento e Feature Engineering**, preparando toda a base de dados de forma robusta e reproduzível para o futuro treinamento dos modelos classificadores.

## 1. Objetivo da Fase 3
O principal objetivo desta etapa é estruturar o pipeline de pré-processamento de dados e engenharia de atributos (Feature Engineering) de forma modular, reutilizável e segura contra vazamento de dados (*data leakage*).

### Principais Diretrizes:
- **Variável Alvo Binária**: `high_quality` = 1 para vinhos com nota `quality >= 7`, e 0 para notas `< 7`.
- **Divisão Estratificada**: Amostragem estratificada (`stratify=y`) devido ao forte desbalanceamento da classe premium.
- **Feature Engineering**: Criação de novas variáveis baseadas em combinados físico-químicos recomendados, utilizando valor *epsilon* de proteção contra divisão por zero.
- **Pipeline Scikit-Learn**: Pipeline integrada com imputação de nulos (`SimpleImputer` por mediana) e escalonamento numérico (`StandardScaler` ou `RobustScaler`).
- **Prevenção de Data Leakage**: O pipeline de pré-processamento é ajustado (*fit*) **exclusivamente** no conjunto de treino, sendo apenas aplicado (*transform*) no conjunto de teste.

## 2. Carregamento dos Dados
Importamos o carregador automatizado da nossa biblioteca modular `src`.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Adiciona o diretório raiz ao path para importação dos pacotes de 'src'
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_wine_data
from src.preprocessing import create_target_variable, split_data, create_preprocessing_pipeline
from src.features import engineer_features

# Carrega a base bruta
df_raw = load_wine_data()
print(f"Dataset carregado. Dimensões: {df_raw.shape[0]} linhas, {df_raw.shape[1]} colunas.")

## 3. Criação da Variável Alvo
Aplicamos a regra de corte para binarizar as avaliações sensoriais de qualidade.

In [ ]:
df_target = create_target_variable(df_raw, source_col="quality", target_col="high_quality")
df_target[["quality", "high_quality"]].head()

## 4. Feature Engineering
Geramos as novas características combinadas físico-químicas de forma segura:

In [ ]:
df_engineered = engineer_features(df_target)
print(f"Novas dimensões pós Feature Engineering: {df_engineered.shape[0]} linhas, {df_engineered.shape[1]} colunas.")
df_engineered[["sulfur_ratio", "acidity_balance", "alcohol_density_ratio", "sugar_alcohol_ratio"]].head()

## 5. Separação Treino/Teste
Separamos as features (`X`) do alvo (`y`), removendo o ID de controle e a coluna de qualidade original. Em seguida, realizamos a divisão em treino (80%) e teste (20%) com estratificação.

In [ ]:
# Identifica a coluna ID caso esteja presente
id_cols = [c for c in ["Id", "id", "ID"] if c in df_engineered.columns]

X_train, X_test, y_train, y_test = split_data(
    df_engineered,
    target_col="high_quality",
    drop_cols=id_cols,
    test_size=0.2,
    random_state=42
)

## 6. Pipeline de Pré-processamento
Validamos o pipeline scikit-learn. O pipeline realiza a imputação mediana de possíveis dados nulos e aplica a padronização numérica. 

> [!IMPORTANT]
> Para garantir que **não ocorra vazamento de dados** (data leakage), o ajuste (*fit*) é executado estritamente em `X_train`.

In [ ]:
# Criamos o pipeline configurando StandardScaler por padrão
preproc_pipeline = create_preprocessing_pipeline(scaler="standard")

# Ajusta e transforma os dados de TREINO
X_train_processed = preproc_pipeline.fit_transform(X_train)

# Apenas transforma os dados de TESTE (sem fit!)
X_test_processed = preproc_pipeline.transform(X_test)

print(f"Dados de treino pré-processados: {X_train_processed.shape}")
print(f"Dados de teste pré-processados: {X_test_processed.shape}")

## 7. Validação dos Outputs e Auditoria
Verificamos se todas as restrições metodológicas obrigatórias foram atendidas com sucesso.

In [ ]:
# 1. Validação de Target Leakage: quality, high_quality e Id não devem estar em X
forbidden_cols = ["quality", "high_quality", "Id", "id", "ID"]
leakage_found = [col for col in forbidden_cols if col in X_train.columns]
assert len(leakage_found) == 0, f"Erro! Colunas proibidas encontradas em X: {leakage_found}"
print("✔ Validação de Vazamento de Alvos: Passou (Nenhum target ou ID está nas features preditoras).")

# 2. Validação da Estratificação: Proporções de classe de treino e teste devem ser coerentes
train_pct = y_train.value_counts(normalize=True).get(1, 0.0) * 100
test_pct = y_test.value_counts(normalize=True).get(1, 0.0) * 100
print(f"✔ Proporção da classe positiva (vinhos premium):")
print(f"   - Base de Treino: {train_pct:.2f}%")
print(f"   - Base de Teste: {test_pct:.2f}%")
assert abs(train_pct - test_pct) < 0.5, "Erro: Desvio de estratificação excessivo!"
print("✔ Validação de Estratificação: Passou (Proporções consistentes entre splits).")

## 8. Treinamento de Modelos (Fase 4)
Nesta seção treinamos os três classificadores principais: Regressão Logística, Random Forest e Gradient Boosting (Hist). O ajuste (*fit*) é feito através de Pipelines do Scikit-Learn que garantem o escalonamento sem vazamento de dados.

In [ ]:
import json
from src.train import train_model

models = {}
model_configs = [
    ("logistic_regression", "standard"),
    ("random_forest", "none"),
    ("gradient_boosting", "none")
]

for model_name, scaler_type in model_configs:
    print(f"\nTreinando {model_name}...")
    pipe, model = train_model(model_name, X_train, y_train, scaler=scaler_type)
    models[model_name] = {"pipeline": pipe, "classifier": model}


## 9. Avaliação Individual e Comparação (Fase 5)
Avaliamos os modelos com foco principal no **F1-Score da Classe 1** (vinhos premium), além de precision, recall e ROC-AUC.

In [ ]:
from src.evaluate import evaluate_classifier, compare_models

results = {}
for name, model_data in models.items():
    print(f"\n--- {name.upper()} ---")
    res = evaluate_classifier(model_data["pipeline"], X_test, y_test)
    results[name] = res
    
print("\n--- COMPARAÇÃO CONSOLIDADA ---")
comparison_df = compare_models(results)


## 10. Visualização de Resultados
Geramos gráficos de performance para auxiliar na interpretação do modelo selecionado.

In [ ]:
from src.plots import plot_confusion_matrices, plot_roc_curves, plot_model_comparison_bar, plot_feature_importance

# Gerando gráficos e salvando na pasta results/figures
plot_confusion_matrices(results, save_path="../results/figures/confusion_matrices.png")
plot_roc_curves(results, save_path="../results/figures/roc_curves.png")
plot_model_comparison_bar(comparison_df, save_path="../results/figures/model_comparison.png")

# Feature Importance para o Random Forest
plot_feature_importance(models["random_forest"]["classifier"], X_train.columns.tolist(), save_path="../results/figures/feature_importance_random.png")


## 11. Conclusão e Serialização do Modelo
O melhor modelo foi selecionado automaticamente com base no F1-Score da Classe 1, serializado e um relatório markdown consolidado foi gerado.

In [ ]:
import joblib
from src.modeling_generator import generate_modeling_report

best_model_name = comparison_df.iloc[0]["Modelo"]
print(f"Melhor Modelo Selecionado: {best_model_name}")

best_pipeline = models[best_model_name]["pipeline"]
joblib.dump(best_pipeline, "../results/models/best_model.pkl")
print("Modelo serializado com sucesso em results/models/best_model.pkl")
